# TEP — De la predicción a la recomendación de ajustes de línea

Objetivo de este notebook: no solo predecir cómo se comporta el proceso (lo que ya sabes hacer con XGBoost), sino **recomendar qué ajuste de variables manipuladas (XMV) lleva a una mejor variable de calidad (XMEAS)**.

Esto es deliberadamente el mismo tipo de problema que vas a tener con la almazara: variables manipulables de línea (setpoints) → variables medidas de proceso → una variable de calidad final. El dataset y los nombres de columna van a cambiar, pero la arquitectura del pipeline no.

**Para usar el TEP real:**
1. Descarga `TEP_FaultFree_Training.csv` de [kaggle.com/datasets/afrniomelo/tep-csv](https://www.kaggle.com/datasets/afrniomelo/tep-csv)
2. Filtra una sola `simulationRun` (p. ej. `df[df['simulationRun'] == 1]`)
3. Cambia `RUTA_CSV`, `VARIABLES_XMV` y `VARIABLES_XMEAS` en la celda de configuración de más abajo por los nombres reales (`xmv_1`...`xmv_11`, `xmeas_1`...`xmeas_41`)

El resto del notebook funciona igual sin tocar nada más — esa es la idea: la arquitectura no depende del dataset concreto.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import itertools

plt.style.use('ggplot')
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 1. Configuración — cambia esto para usar tus propios datos

Esta celda es la única que deberías necesitar tocar al cambiar de dataset (simulado → TEP real → datos de la almazara). Todo lo demás está escrito en términos de estas variables.

In [ ]:
RUTA_CSV = '../Datasets/TEP_FaultFree_Training.csv'

# Columna de orden temporal (equivalente a 'sample' en TEP)
COLUMNA_TIEMPO = 'sample'

# La diferencia entre dos samples en tiempo es de 3 minutos

df_raw = pd.read_csv(RUTA_CSV)
df_raw = df_raw.sort_values(COLUMNA_TIEMPO).reset_index(drop=True)

X_dict = {
'XMEAS_1':'A_feed_stream',
'XMEAS_2':'D_feed_stream',
'XMEAS_3':'E_feed_stream',
'XMEAS_4':'Total_fresh_feed_stripper',
'XMEAS_5':'Recycle_flow_into_rxtr',
'XMEAS_6':'Reactor_feed_rate',
'XMEAS_7':'Reactor_pressure',
'XMEAS_8':'Reactor_level',
'XMEAS_9':'Reactor_temp',
'XMEAS_10':'Purge_rate',
'XMEAS_11':'Separator_temp',
'XMEAS_12':'Separator_level',
'XMEAS_13':'Separator_pressure',
'XMEAS_14':'Separator_underflow',
'XMEAS_15':'Stripper_level',
'XMEAS_16':'Stripper_pressure',
'XMEAS_17':'Stripper_underflow',
'XMEAS_18':'Stripper_temperature',
'XMEAS_19':'Stripper_steam_flow',
'XMEAS_20':'Compressor_work',
'XMEAS_21':'Reactor_cooling_water_outlet_temp',
'XMEAS_22':'Condenser_cooling_water_outlet_temp',
'XMEAS_23':'Composition_of_A_rxtr_feed',
'XMEAS_24':'Composition_of_B_rxtr_feed',
'XMEAS_25':'Composition_of_C_rxtr_feed',
'XMEAS_26':'Composition_of_D_rxtr_feed',
'XMEAS_27':'Composition_of_E_rxtr_feed',
'XMEAS_28':'Composition_of_F_rxtr_feed',
'XMEAS_29':'Composition_of_A_purge',
'XMEAS_30':'Composition_of_B_purge',
'XMEAS_31':'Composition_of_C_purge',
'XMEAS_32':'Composition_of_D_purge',
'XMEAS_33':'Composition_of_E_purge',
'XMEAS_34':'Composition_of_F_purge',
'XMEAS_35':'Composition_of_G_purge',
'XMEAS_36':'Composition_of_H_purge',
'XMEAS_37':'Composition_of_D_product',
'XMEAS_38':'Composition_of_E_product',
'XMEAS_39':'Composition_of_F_product',
'XMEAS_40':'Composition_of_G_product',
'XMEAS_41':'Composition_of_H_product',
'XMV_1':'D_feed_flow_valve',
'XMV_2':'E_feed_flow_valve',
'XMV_3':'A_feed_flow_valve',
'XMV_4':'Total_feed_flow_stripper_valve',
'XMV_5':'Compressor_recycle_valve',
'XMV_6':'Purge_valve',
'XMV_7':'Separator_pot_liquid_flow_valve',
'XMV_8':'Stripper_liquid_product_flow_valve',
'XMV_9':'Stripper_steam_valve',
'XMV_10':'Reactor_cooling_water_flow_valve',
'XMV_11':'Condenser_cooling_water_flow_valve',
'XMV_12':'Agitator_speed'
   }

# df_raw = df_raw.rename(columns = lambda x:X_dict[x.upper()] if x.upper() in X_dict.keys()  else x)

print("Dimensiones:", df_raw.shape)
df_raw.head()

# Obtener las filas que pertenecen a la simulación 1
df_sim = df_raw[df_raw['simulationRun'] == 1]

## 2. Cargar datos y vistazo rápido

In [ ]:
# Usa X_dict para renombrar las columnas de df_raw a nombres más descriptivos. Esto facilita la interpretación de los datos y la identificación de las variables relevantes para el análisis.

# Variable de calidad objetivo — lo que quieres optimizar (análogo a acidez/fenoles)
VARIABLE_CALIDAD = 'xmeas_7'

# Variables manipulables (lo que un operador podría ajustar) — tus "XMV"
VARIABLES_XMV = [meas for meas in df_sim.columns if meas.upper() in X_dict.keys() and meas.startswith('xmv_') and meas != VARIABLE_CALIDAD]

# Variables de proceso medidas (sensores) — tus "XMEAS" intermedias
VARIABLES_XMEAS = [meas for meas in df_sim.columns if meas.upper() in X_dict.keys() and meas.startswith('xmeas_') and meas != VARIABLE_CALIDAD]

print("Variables de calidad objetivo:", VARIABLE_CALIDAD)
print("Variables manipulables:", VARIABLES_XMV)
print("Variables de proceso medidas:", VARIABLES_XMEAS)


In [ ]:
fig, axes = plt.subplots(len(VARIABLES_XMV) + len(VARIABLES_XMEAS) + 1, 1,
                          figsize=(14, 2.2 * (len(VARIABLES_XMV) + len(VARIABLES_XMEAS) + 1)),
                          sharex=True)

todas_vars = VARIABLES_XMV + VARIABLES_XMEAS + [VARIABLE_CALIDAD]
colores = plt.cm.tab10(np.linspace(0, 1, len(todas_vars)))

for ax, var, color in zip(axes, todas_vars, colores):
    ax.plot(df_sim[COLUMNA_TIEMPO], df_sim[var], color=color)
    ax.set_ylabel(var, fontsize=8)

axes[-1].set_xlabel(COLUMNA_TIEMPO)
plt.suptitle('Variables manipuladas (XMV), medidas (XMEAS) y calidad')
plt.tight_layout()
plt.show()

## 3. Autocorrelación — cuántos lags tienen sentido aquí

Igual que hicimos con el dataset de la bomba, pero esta vez con un proceso de dinámica más lenta (más inercia). Espera ver el PACF de la variable de calidad cayendo dentro de la banda mucho más tarde que en el caso de la bomba — eso confirma que aquí sí puede tener sentido un rango de lags más largo, o incluso el espaciado por potencias que descartamos antes.

In [ ]:
# n_lags_analisis = 25
# variables_pacf = VARIABLES_XMEAS + [VARIABLE_CALIDAD]

# fig, axes = plt.subplots(len(variables_pacf), 2, figsize=(14, 4 * len(variables_pacf)))
# if len(variables_pacf) == 1:
#     axes = axes.reshape(1, -1)

# for i, var in enumerate(variables_pacf):
#     plot_acf(df_raw[var], lags=n_lags_analisis, ax=axes[i, 0])
#     axes[i, 0].set_title(f'ACF - {var}')
#     plot_pacf(df_raw[var], lags=n_lags_analisis, ax=axes[i, 1], method='ywm')
#     axes[i, 1].set_title(f'PACF - {var}')

# plt.tight_layout()
# plt.show()

## 4. Feature engineering (reutilizamos la función ya construida)

Misma función que en el notebook anterior, con lags + rolling + diff, causales.

In [ ]:
def _lags_potencia(n_lags_max, base=2):
    lags = []
    l = 1
    while l < n_lags_max:
        lags.append(l)
        l *= base
    lags.append(n_lags_max)
    return sorted(set(lags))


def construir_features(df, variables_entrada, usar_lags=True, n_lags=3,
                        espaciado_lags='consecutivo', base_potencia=2,
                        usar_rolling_mean=False, ventana_mean=5,
                        usar_rolling_std=False, ventana_std=5,
                        usar_diff=False):
    df_out = df.copy()
    columnas_features = list(variables_entrada)

    if espaciado_lags == 'potencia':
        lista_lags = _lags_potencia(n_lags, base=base_potencia)
    else:
        lista_lags = list(range(1, n_lags + 1))

    for var in variables_entrada:
        if usar_lags:
            for lag in lista_lags:
                col = f'{var}_lag_{lag}'
                df_out[col] = df_out[var].shift(lag)
                columnas_features.append(col)
        if usar_rolling_mean:
            col = f'{var}_roll_mean_{ventana_mean}'
            df_out[col] = df_out[var].shift(1).rolling(window=ventana_mean).mean()
            columnas_features.append(col)
        if usar_rolling_std:
            col = f'{var}_roll_std_{ventana_std}'
            df_out[col] = df_out[var].shift(1).rolling(window=ventana_std).std()
            columnas_features.append(col)
        if usar_diff:
            col = f'{var}_diff'
            df_out[col] = df_out[var].diff()
            columnas_features.append(col)

    return df_out, columnas_features


def evaluar_modelo(X, y, train_frac=0.7, n_estimators=150, max_depth=3, learning_rate=0.1):
    corte = int(len(X) * train_frac)
    X_train, X_test = X.iloc[:corte], X.iloc[corte:]
    y_train, y_test = y.iloc[:corte], y.iloc[corte:]

    modelo = xgb.XGBRegressor(
        n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate,
        random_state=42, n_jobs=-1
    )
    modelo.fit(X_train, y_train)
    preds = modelo.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    return {'modelo': modelo, 'rmse': rmse, 'X_test': X_test, 'y_test': y_test,
            'preds': preds, 'corte': corte}

## 5. Modelo predictivo de la variable de calidad

Antes de recomendar nada, necesitas un modelo que prediga bien la variable de calidad a partir de las variables manipuladas y medidas. **Este modelo es el corazón de todo lo que viene después** — si predice mal, cualquier recomendación que construyas encima es ruido con apariencia de decisión informada.

Ojo: aquí metemos también las XMEAS intermedias como *inputs*, no solo las XMV. Esto es una decisión de diseño explícita — asumimos que en el momento de "recomendar" ya conoces el estado actual del proceso (temperaturas, presiones), y lo que decides es el próximo ajuste de XMV. Si tu caso real es distinto (recomendar sin conocer el estado intermedio), tendrías que quitar las XMEAS de la lista de entrada.

In [ ]:
variables_entrada_calidad = VARIABLES_XMV + VARIABLES_XMEAS

df_feat, cols_feat = construir_features(
    df_sim, variables_entrada_calidad,
    usar_lags=True, n_lags=5, espaciado_lags='consecutivo',
    usar_rolling_mean=True, ventana_mean=5,
    usar_rolling_std=False,
    usar_diff=True
)
df_feat[VARIABLE_CALIDAD] = df_sim[VARIABLE_CALIDAD].values
df_feat = df_feat.dropna().reset_index(drop=True)

X_calidad = df_feat[cols_feat]
y_calidad = df_feat[VARIABLE_CALIDAD]

res_calidad = evaluar_modelo(X_calidad, y_calidad, n_estimators=200, max_depth=4)
print(f"RMSE prediciendo {VARIABLE_CALIDAD}: {res_calidad['rmse']:.4f}")
print(f"Rango real de la variable: [{y_calidad.min():.2f}, {y_calidad.max():.2f}]  (desviación típica: {y_calidad.std():.2f})")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(res_calidad['y_test'].values, label='Real', color='blue', alpha=0.6)
plt.plot(res_calidad['preds'], label='Predicción', color='red', linestyle='dashed', alpha=0.8)
plt.title(f"Modelo de calidad — RMSE {res_calidad['rmse']:.3f}")
plt.xlabel('Muestras de test')
plt.ylabel(VARIABLE_CALIDAD)
plt.legend()
plt.tight_layout()
plt.show()

xgb.plot_importance(res_calidad['modelo'], height=0.5, importance_type='weight', max_num_features=12)
plt.title('Importancia de variables — modelo de calidad')
plt.tight_layout()
plt.show()

## 6. De predicción a recomendación

Aquí está la pieza nueva. La idea:

1. Tomamos el **estado actual** del proceso (última fila de datos: lags, rolling, etc. ya calculados salvo las XMV, que son lo que queremos decidir).
2. Probamos una **rejilla de combinaciones posibles de XMV**, dentro del rango que el proceso ya ha mostrado en los datos históricos (nunca fuera — el árbol no sabe extrapolar).
3. Para cada combinación, reconstruimos el vector de features y le pedimos al modelo que prediga la calidad resultante.
4. Nos quedamos con la combinación que maximiza (o minimiza, según el objetivo) la calidad predicha.

### La advertencia que de verdad importa aquí

Esto **no es optimización real de proceso** — es "probar muchas combinaciones ya vistas y quedarte con la mejor según lo que el modelo cree". Si el óptimo real está fuera del rango histórico, este método nunca lo va a encontrar, y si lo fuerzas a buscar fuera de rango, el modelo va a inventarse una predicción sin ninguna base real. Por eso la rejilla está anclada estrictamente al min/max observado — es una salvaguarda, no un capricho.

In [ ]:
def construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_entrada, idx=-1):
    """
    Toma la última fila disponible de df_feat (con todos los lags/rolling/diff ya calculados)
    como snapshot del estado del proceso, para usarla como plantilla en la búsqueda.
    """
    return df_feat[cols_feat].iloc[[idx]].copy()


def recomendar_ajuste(modelo, df_raw, df_feat, cols_feat, variables_xmv, variable_calidad,
                       modo='rejilla', n_puntos_por_variable=5, n_muestras_aleatorias=2000,
                       objetivo='maximizar', seed=42):
    """
    Busca la mejor combinación de XMV, ACOTADA al rango histórico observado (nunca extrapola).

    modo='rejilla': malla completa (itertools.product). Exhaustivo pero crece como
                     n_puntos_por_variable ** n_variables -- inviable con muchas variables
                     (con 11 XMV y 5 puntos/variable son ~49 millones de combinaciones).
                     Útil solo con pocas variables (hasta 4-5) o pocos puntos por variable.

    modo='aleatorio': muestrea n_muestras_aleatorias combinaciones al azar dentro del rango
                       válido de cada variable. El coste NO depende del número de variables,
                       solo del número de muestras que decidas pagar -- por eso escala a
                       cualquier número de XMV sin explotar en memoria ni en tiempo.
                       En espacios de muchas dimensiones, random search suele acercarse al
                       óptimo de grid search con una fracción ínfima del coste, porque la
                       mayoría de combinaciones de una malla completa son redundantes.

    IMPORTANTE: en ambos modos, todas las combinaciones se construyen de golpe en un único
    DataFrame y se evalúan con UNA sola llamada a modelo.predict() (vectorizado), no una
    llamada por combinación -- eso es lo que evita el overhead fijo de predict() multiplicado
    por miles/millones de veces, que es la causa más común de lentitud aquí (no el tamaño
    del modelo, ni la falta de GPU).
    """
    estado_base = construir_vector_estado_actual(df_raw, df_feat, cols_feat, variables_xmv)
    rng = np.random.default_rng(seed)

    rangos = {var: (df_raw[var].min(), df_raw[var].max()) for var in variables_xmv}

    if modo == 'rejilla':
        rejillas = {var: np.linspace(vmin, vmax, n_puntos_por_variable)
                    for var, (vmin, vmax) in rangos.items()}
        combinaciones = np.array(list(itertools.product(*[rejillas[v] for v in variables_xmv])))
    elif modo == 'aleatorio':
        combinaciones = np.column_stack([
            rng.uniform(rangos[var][0], rangos[var][1], n_muestras_aleatorias)
            for var in variables_xmv
        ])
    else:
        raise ValueError("modo debe ser 'rejilla' o 'aleatorio'")

    n_combos = len(combinaciones)

    # Construir TODAS las filas de una vez (vectorizado), en vez de una por iteración
    filas = pd.concat([estado_base] * n_combos, ignore_index=True)
    for j, var in enumerate(variables_xmv):
        if var in filas.columns:
            filas[var] = combinaciones[:, j]

    # UNA sola llamada a predict() para todas las combinaciones
    preds = modelo.predict(filas[cols_feat])

    df_resultados = pd.DataFrame(combinaciones, columns=variables_xmv)
    df_resultados['calidad_predicha'] = preds

    ascending = (objetivo == 'minimizar')
    df_resultados = df_resultados.sort_values('calidad_predicha', ascending=ascending).reset_index(drop=True)
    return df_resultados

### Ejecutar la recomendación

Buscamos la combinación de XMV que **maximiza** la calidad predicha (cambia `objetivo='minimizar'` si tu variable de calidad es del tipo "cuanto menos, mejor" — como sería el caso de acidez).

In [ ]:
# Con 11 variables XMV reales, modo='rejilla' con 5 puntos/variable son ~48.8 millones
# de combinaciones -- eso es lo que te mató el kernel. Con >= 5-6 variables, usa
# modo='aleatorio': el coste no depende de cuántas XMV tengas, solo de cuántas
# muestras decidas pagar (ver benchmark en la siguiente celda para comparar ambos
# modos con un tamaño de rejilla que SÍ es seguro, y ver por qué el real no lo era).
df_recomendaciones = recomendar_ajuste(
    modelo=res_calidad['modelo'],
    df_raw=df_raw,
    df_feat=df_feat,
    cols_feat=cols_feat,
    variables_xmv=VARIABLES_XMV,
    variable_calidad=VARIABLE_CALIDAD,
    modo='aleatorio',
    n_muestras_aleatorias=2000,
    objetivo='maximizar'
)

print("1 combinaciones recomendadas:")
df_recomendaciones.head(1)

### Por qué esto ya no "tarda una barbaridad": rejilla vs aleatorio, medido

Dos cosas cambiaron respecto a la primera versión: (1) `predict()` se llama **una sola vez** con todas las combinaciones juntas en vez de una vez por combinación, y (2) puedes elegir `modo='aleatorio'` para que el coste no explote con el número de variables. Vamos a medirlo con tus propios datos y modelo, no de oídas.

In [ ]:
import time

n_xmv = len(VARIABLES_XMV)
# Con muchas variables, n_puntos_por_variable pequeño para que la rejilla no reviente
# (5 puntos ^ 11 variables = 48.8M combinaciones -- eso es justo lo que mató el kernel arriba).
n_puntos_seguro = 2 if n_xmv >= 6 else 5

for modo, kwargs in [
    ('rejilla', {'n_puntos_por_variable': n_puntos_seguro}),
    ('aleatorio', {'n_muestras_aleatorias': 500}),
]:
    t0 = time.time()
    df_bench = recomendar_ajuste(
        modelo=res_calidad['modelo'], df_raw=df_raw, df_feat=df_feat, cols_feat=cols_feat,
        variables_xmv=VARIABLES_XMV, variable_calidad=VARIABLE_CALIDAD, modo=modo, objetivo='maximizar',
        **kwargs
    )
    t1 = time.time()
    mejor_calidad = df_bench.iloc[0]["calidad_predicha"]
    print(f"modo={modo:10s}  combinaciones={len(df_bench):8d}  tiempo={t1-t0:.4f}s  mejor_calidad={mejor_calidad:.4f}")

print()
print(f"Con tus {n_xmv} variables XMV reales:")
print(f"  modo=rejilla con 5 puntos/variable    -> {5**n_xmv:,} combinaciones  <- esto te mató el kernel arriba")
print(f"  modo=aleatorio con 2000 muestras       -> 2,000 combinaciones (mismo coste siempre, no depende de n_xmv)")

## 7. Comprobación de fiabilidad — ¿la recomendación está dentro de lo que el modelo realmente conoce?

Esta celda es la que convierte "una recomendación" en "una recomendación en la que puedes confiar un poco más". Comprobamos si la mejor combinación encontrada corresponde a una zona del espacio de XMV donde hubo **suficientes datos históricos parecidos** — si la mejor recomendación vive en una esquina del espacio con pocos puntos de entrenamiento cerca, es una señal de alerta, no una señal de éxito.

In [ ]:
from scipy.spatial import cKDTree

def distancia_a_datos_historicos(mejor_combo, df_raw, variables_xmv):
    """
    Distancia (normalizada) de la combinación recomendada al punto histórico más cercano,
    usando un KDTree en vez de una matriz de distancias completa (cdist).

    Con un histórico grande (aquí, ~250.000 filas si df_raw son las 500 simulaciones
    completas), calcular todas las distancias contra todas las filas no es necesario --
    solo buscamos EL vecino más cercano, que es justo lo que un KDTree resuelve sin
    construir esa matriz gigante en memoria.
    """
    datos_hist = df_raw[variables_xmv].values
    rangos = datos_hist.max(axis=0) - datos_hist.min(axis=0)
    rangos[rangos == 0] = 1
    datos_hist_norm = datos_hist / rangos
    combo_norm = np.array(mejor_combo) / rangos

    tree = cKDTree(datos_hist_norm)
    dist_min, _ = tree.query(combo_norm)
    return dist_min

mejor_combo = df_recomendaciones.iloc[0][VARIABLES_XMV].values
dist_min = distancia_a_datos_historicos(mejor_combo, df_raw, VARIABLES_XMV)

# Referencia: distancia típica entre puntos históricos consecutivos, para dar contexto a la cifra.
# Aquí NO usamos cdist (que compararía todas las filas contra todas -> misma explosión de
# memoria que arriba). Solo queremos la distancia fila[i] vs fila[i+1], una comparación
# uno-a-uno, así que una resta vectorizada basta y es muchísimo más barata.
datos_hist_norm_ref = df_raw[VARIABLES_XMV].values / (df_raw[VARIABLES_XMV].max() - df_raw[VARIABLES_XMV].min()).values
dist_referencia = np.median(np.linalg.norm(datos_hist_norm_ref[1:] - datos_hist_norm_ref[:-1], axis=1))

print(f"Mejor combinación recomendada: {dict(zip(VARIABLES_XMV, mejor_combo))}")
calidad_predicha_top = df_recomendaciones.iloc[0]["calidad_predicha"]
print(f"Calidad predicha: {calidad_predicha_top:.4f}")
print(f"Distancia (normalizada) al dato histórico más cercano: {dist_min:.4f}")
print(f"Distancia típica de referencia entre muestras consecutivas: {dist_referencia:.4f}")

if dist_min > 3 * dist_referencia:
    print("\n⚠️  ALERTA: esta recomendación está lejos de cualquier combinación vista en los datos.")
    print("   El modelo está extrapolando -- trátala como hipótesis a validar, no como instrucción a aplicar.")
else:
    print("\n✓ La recomendación está razonablemente cerca de zonas ya observadas en los datos históricos.")

## 8. Qué falta para que esto sea un sistema de recomendación real (no un ejercicio)

Deja esto anotado para cuando tengas los datos reales de la almazara, porque son las piezas que este notebook simplifica a propósito para que puedas centrarte en la arquitectura:

- **La variable de calidad real puede llegar con retardo y a otra cadencia** (análisis de laboratorio vs sensores en línea). Aquí la calidad se genera al mismo ritmo que el resto — en tu caso probablemente no será así, y tendrás que resolver un forecasting de horizonte largo antes de poder recomendar nada.
- **La búsqueda en rejilla no escala bien con muchas variables manipulables.** Con 3 XMV y 5 puntos por variable ya son 125 combinaciones; con 11 XMV (como en el TEP real) serían millones. Para eso existen métodos de optimización más eficientes (optimización bayesiana, algoritmos genéticos) — pero no los necesitas para aprender el concepto.
- **El modelo no sabe nada de restricciones de seguridad o de proceso** (p. ej. "la presión no puede superar X"). Cualquier recomendación real necesita una capa de reglas duras por encima del modelo, no solo el óptimo estadístico.
- **La comprobación de distancia a datos históricos es una heurística simple**, no una medida de incertidumbre real del modelo. Para algo más riguroso, XGBoost puede combinarse con estimación de incertidumbre (ensembles, quantile regression) — es un paso natural una vez domines esta versión básica.
- **Sobre GPU**: acelera el *entrenamiento* de XGBoost (`tree_method='hist', device='cuda'`) sobre datasets grandes, y `predict()` vectorizado sobre muchas filas a la vez. No ayuda si el cuello de botella es cómo está escrito el bucle de búsqueda (llamadas individuales a predict, o generar una malla que no cabe en memoria) — eso se arregla vectorizando y con `modo='aleatorio'`, como en este notebook, no añadiendo hardware. Si algún día entrenas con millones de filas y el *entrenamiento* (no la recomendación) es lo lento, ahí sí GPU tiene sentido.
- **Si más adelante necesitas exprimir más la búsqueda que random search**, el siguiente escalón es optimización bayesiana (p. ej. librería `scikit-optimize` o `optuna`): en vez de muestrear a ciegas, usa los resultados ya evaluados para decidir qué combinación probar después, así que converge a un buen óptimo con muchas menos llamadas al modelo que random search. No lo necesitas todavía con pocas variables, pero es la pieza que te faltaría si el espacio de búsqueda creciera mucho más o cada evaluación del modelo fuera cara.